# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [1]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipyleaflet'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [2]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [3]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [4]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    description = f"{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(description, location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

Determinants of Kansas Farmland Transaction Frequency<br />2023 Agricultural & Applied Economics Association Annual Meeting; Washington D.C., USA Washington, District of Columbia, United States


Do Farmers Capitalize Conservation Practices in Their Own Land Valuations?<br />2026 Agricultural & Applied Economics Association Annual Meeting; Kansas City, MO, USA Kansas City, Jackson County, Missouri, United States
New York Farmland Market Trends and the Impacts of Renewable Energy<br />Broome County Cornell Cooperative Extension Realtor Education Day; Binghamton, NY, USA City of Binghamton, Broome County, New York, 13905, United States


Farm or Fortune? The Role of Heirs' Intentions and Tax Policies in Farmland Succession Planning<br />2025 Agricultural & Applied Economics Association Annual Meeting; Denver, CO, USA Denver, Colorado, United States


Northeast Farmland Market Update<br />New York State Assessors Association, Cornell Seminar on Professional Development; Ithaca, NY, USA Town of Ithaca, Tompkins County, New York, United States


Do Farmers Capitalize Conservation Practices in Their Own Land Valuations?<br />2026 Midwest Extension Farm Management and Outlook Conference; Lexington, KY, USA Lexington, Fayette County, Kentucky, United States


Community Property, Taxes, and Estate Planning: How marital property regimes affect farmland markets and the lock-in effect<br />Forest and Rangeland Taxation Meeting; Durham, NC, USA Durham, Durham County, North Carolina, United States
Community Property, Taxes, and Estate Planning: How marital property regimes affect farmland markets and the lock-in effect<br />2026 Agricultural & Applied Economics Association Annual Meeting; Kansas City, MO, USA Kansas City, Jackson County, Missouri, United States
Familial Intentions: Farmers' Farmland Decisions and Succession Planning Under Uncertainty<br />SCC-76: Economics and Management of Risk in Agriculture and Natural Resources; Kansas City, MO, USA Kansas City, Jackson County, Missouri, United States


The Implications of Farm Structure and Tax Policies for Estate Planning<br />NC-1177: Agricultural and Rural Finance Markets in Transition; Omaha, NE, USA Omaha, Douglas County, Nebraska, United States
Agricultural Land Turnover and Why it Matters<br />Joint Meeting of SCC-76: Economics and Management of Risk in Agriculture and Natural Resources and NC-1177: Agricultural and Rural Finance Markets in Transition; Kansas City, MO, USA Kansas City, Jackson County, Missouri, United States
The Relationship Between Backyard Animal Keeping and Food Security Status in the United States<br />2025 Agricultural & Applied Economics Association Annual Meeting; Denver, CO, USA Denver, Colorado, United States
Modeling U.S. Sorghum Export Destinations During Multiple Trade Shocks of the U.S.-China Trade War<br />SCC-76: Economics and Management of Risk in Agriculture and Natural Resources; Kansas City, MO, USA Kansas City, Jackson County, Missouri, United States
Community Property, Taxes, and Estate Pl

In [5]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

# Re-center the generated map over the continental U.S. instead of getorg's
# default world view, since getorg doesn't expose center/zoom as a parameter.
map_html_path = "talkmap/map.html"
with open(map_html_path, "r") as f:
    map_html = f.read()

map_html = map_html.replace(
    "latlng = L.latLng(30, 10);", "latlng = L.latLng(39.8, -98.6);"
).replace(
    "zoom: 0.7", "zoom: 3"
).replace(
    "maxClusterRadius: 80", "maxClusterRadius: 30"
).replace(
    "<span>Mouse over a cluster to see the bounds of its children and click a cluster to zoom to those bounds</span>",
    '<p style="font-size: smaller; color: var(--global-text-color-light);">Click a cluster to zoom in and see the individual talks.</p>'
)

with open(map_html_path, "w") as f:
    f.write(map_html)